# Representation Learning

Installing dependencies

In [1]:
!pip install rank_bm25 sentence-transformers datasets datasets tf-keras transformers[torch] -q # quiet install

ERROR: ld.so: object '/opt/conda/lib/libmkl_def.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_avx2.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_core.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_intel_lp64.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_intel_thread.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_def.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_avx2.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_core.so' from LD_PRE

In [2]:
import numpy as np
import pandas as pd

In [3]:
PATH_COLLECTION_DATA = "subtask4b_collection_data.pkl"
PATH_QUERY_TRAIN_DATA = "subtask4b_query_tweets_train.tsv"
PATH_QUERY_DEV_DATA = "subtask4b_query_tweets_dev.tsv"

df_collection = pd.read_pickle(PATH_COLLECTION_DATA)
df_query_train = pd.read_csv(PATH_QUERY_TRAIN_DATA, sep = '\t')
df_query_dev = pd.read_csv(PATH_QUERY_DEV_DATA, sep = '\t')

In [4]:
df_collection.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7718 entries, 162 to 1056448
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   cord_uid          7718 non-null   object        
 1   source_x          7718 non-null   object        
 2   title             7718 non-null   object        
 3   doi               7677 non-null   object        
 4   pmcid             4959 non-null   object        
 5   pubmed_id         6233 non-null   object        
 6   license           7718 non-null   object        
 7   abstract          7718 non-null   object        
 8   publish_time      7715 non-null   object        
 9   authors           7674 non-null   object        
 10  journal           6668 non-null   object        
 11  mag_id            0 non-null      float64       
 12  who_covidence_id  528 non-null    object        
 13  arxiv_id          20 non-null     object        
 14  label             7718 n

In [5]:
df_collection.head()

,cord_uid,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,mag_id,who_covidence_id,arxiv_id,label,time,timet
162,umvrwgaw,PMC,Professional and Home-Made Face Masks Reduce E...,10.1371/journal.pone.0002618,PMC2440799,18612429,cc-by,BACKGROUND: Governments are preparing for a po...,2008-07-09,"van der Sande, Marianne; Teunis, Peter; Sabel,...",PLoS One,NaN,NaN,NaN,umvrwgaw,2008-07-09,1215561600
611,spiud6ok,PMC,The Failure of R (0),10.1155/2011/527610,PMC3157160,21860658,cc-by,"The basic reproductive ratio, R (0), is one of...",2011-08-16,"Li, Jing; Blakeley, Daniel; Smith?, Robert J.",Comput Math Methods Med,NaN,NaN,NaN,spiud6ok,2011-08-16,1313452800
918,aclzp3iy,PMC,Pulmonary sequelae in a patient recovered from...,10.4103/0970-2113.99118,PMC3424870,22919170,cc-by-nc-sa,The pandemic of swine flu (H1N1) influenza spr...,2012,"Singh, Virendra; Sharma, Bharat Bhushan; Patel...",Lung India,NaN,NaN,NaN,aclzp3iy,2012-01-01,1325376000
993,ycxyn2a2,PMC,What was the primary mode of smallpox transmis...,10.3389/fcimb.2012.00150,PMC3509329,23226686,cc-by,The mode of infection transmission has profoun...,2012-11-29,"Milton, Donald K.",Front Cell Infect Microbiol,NaN,NaN,NaN,ycxyn2a2,2012-11-29,1354147200
1053,zxe95qy9,PMC,"Lessons from the History of Quarantine, from P...",10.3201/eid1902.120312,PMC3559034,23343512,no-cc,"In the new millennium, the centuries-old strat...",2013-02-03,"Tognotti, Eugenia",Emerg Infect Dis,NaN,NaN,NaN,zxe95qy9,2013-02-03,1359849600


In [6]:
df_query_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12853 entries, 0 to 12852
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   post_id     12853 non-null  int64 
 1   tweet_text  12853 non-null  object
 2   cord_uid    12853 non-null  object
dtypes: int64(1), object(2)
memory usage: 301.4+ KB


In [7]:
df_query_train.head()

,post_id,tweet_text,cord_uid
0,0,Oral care in rehabilitation medicine: oral vul...,htlvpvz5
1,1,this study isn't receiving sufficient attentio...,4kfl29ul
2,2,"thanks, xi jinping. a reminder that this study...",jtwb17u8
3,3,Taiwan - a population of 23 million has had ju...,0w9k8iy1
4,4,Obtaining a diagnosis of autism in lower incom...,tiqksd69


In [8]:
df_query_dev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1400 entries, 0 to 1399
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   post_id     1400 non-null   int64 
 1   tweet_text  1400 non-null   object
 2   cord_uid    1400 non-null   object
dtypes: int64(1), object(2)
memory usage: 32.9+ KB


In [9]:
df_query_dev.head()

,post_id,tweet_text,cord_uid
0,16,covid recovery: this study from the usa reveal...,3qvh482o
1,69,"""Among 139 clients exposed to two symptomatic ...",r58aohnu
2,73,I recall early on reading that researchers who...,sts48u9i
3,93,You know you're credible when NIH website has ...,3sr2exq9
4,96,Resistance to antifungal medications is a grow...,ybwwmyqy


## Helper function

In [10]:
import multiprocessing as mp

def parallel_apply(data, func, n_jobs=None):
    if n_jobs is None:
        n_jobs = mp.cpu_count()

    with mp.Pool(n_jobs) as pool:
        result = pool.map(func, data)

    return result

# 2) Setup BM25

In [11]:
from rank_bm25 import BM25Okapi

# Create the BM25 corpus
corpus = df_collection[:][['title', 'abstract']].apply(lambda x: f"{x['title']} {x['abstract']}", axis=1).tolist()
cord_uids = df_collection[:]['cord_uid'].tolist()
tokenized_corpus = [doc.split(' ') for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def get_top_cord_uids(query):
  text2bm25top = {}
  if query in text2bm25top.keys():
      return text2bm25top[query]
  else:
      tokenized_query = query.split(' ')
      doc_scores = bm25.get_scores(tokenized_query)
      indices = np.argsort(-doc_scores)[:5]
      bm25_topk = [cord_uids[x] for x in indices]

      text2bm25top[query] = bm25_topk
      return bm25_topk

In [12]:
df_query_train['bm25_topk'] = parallel_apply(
    df_query_train['tweet_text'],
    get_top_cord_uids,
    n_jobs=None  # or mp.cpu_count() - 1
)
df_query_train.to_pickle("df_query_train.pkl") # only needed once

## Evaluation

In [13]:
# Evaluate retrieved candidates using MRR@k
def get_performance_mrr(data, col_gold, col_pred, list_k = [1, 5, 10]):
    d_performance = {}
    for k in list_k:
        data["in_topx"] = data.apply(lambda x: (1/([i for i in x[col_pred][:k]].index(x[col_gold]) + 1) if x[col_gold] in [i for i in x[col_pred][:k]] else 0), axis=1)
        #performances.append(data["in_topx"].mean())
        d_performance[k] = data["in_topx"].mean()
    return d_performance

results_train = get_performance_mrr(df_query_train, 'cord_uid', 'bm25_topk')
# Printed MRR@k results in the following format: {k: MRR@k}
print(f"Results on the train set: {results_train}")

Results on the train set: {1: 0.5080525947249669, 5: 0.5509388210275163, 10: 0.5509388210275163}


# Representation Learning Approach

## Multiple Negtaives Ranking Loss

In [30]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer


from sentence_transformers import InputExample

train_examples = []

for index, row in df_query_train.iterrows():
    paper_id = row['cord_uid']
    tweet_text = row['tweet_text']
    pos_title = df_collection.loc[df_collection['cord_uid'] == paper_id, 'title'].values[0]
    pos_abstract = df_collection.loc[df_collection['cord_uid'] == paper_id, 'abstract'].values[0]
    pos_text = f"{pos_title}. {pos_abstract}"
    input_example = InputExample(texts=[tweet_text, pos_text], label=1.0)
    train_examples.append(input_example)

    hard_negatives = [cid for cid in row['bm25_topk'] if cid!=row['cord_uid']]
    for neg_id in hard_negatives:
        neg_title = df_collection.loc[df_collection['cord_uid'] == neg_id, 'title'].values[0]
        neg_abstract = df_collection.loc[df_collection['cord_uid'] == neg_id, 'abstract'].values[0]
        neg_text = f"{neg_title}. {neg_abstract}"
    
        input_example = InputExample(texts=[tweet_text, neg_text], label=0.0)
        train_examples.append(input_example)

pd.DataFrame(train_examples).to_pickle("train_examples.pkl")

In [15]:
from torch.utils.data import DataLoader
from sentence_transformers import losses

model = SentenceTransformer('all-MiniLM-L6-v2') # smaller faster transformer

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [26]:
model_filename = "MiniLM-mnr-ep3"

if False:
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=3,
        warmup_steps=100,
        show_progress_bar=True
    )
    model.save(model_filename)
    
model = SentenceTransformer(model_filename)

AttributeError: 'DataFrame' object has no attribute 'read_pickle'

In [36]:
#train_examples = list(pd.read_pickle("train_examples.pkl")[0])
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

model_mpnet = SentenceTransformer('all-mpnet-base-v2')
train_loss = losses.MultipleNegativesRankingLoss(model=model_mpnet)

model_filename = "Mpnet-mnr-ep3"

if True:
    model_mpnet.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=3,
        warmup_steps=100,
        show_progress_bar=True
    )
    model_mpnet.save(model_filename)
    
model_mpnet = SentenceTransformer(model_filename)

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
ERROR: ld.so: object '/opt/conda/lib/libmkl_def.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_avx2.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_core.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_intel_lp64.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_intel_thread.so' from LD_PRELOAD

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


KeyboardInterrupt: 

In [17]:
paper_list = df_collection[:][['title', 'abstract']].apply(lambda x: f"{x['title']}. {x['abstract']}", axis=1).tolist()
cord_uids = df_collection[:]['cord_uid'].tolist()

paper_embeddings = model.encode(paper_list)  # List of paper titles + abstracts

In [39]:
# Use cosine similarity to find top match
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_top_cord_uids_neural(query, model, top_k=5):
    query_embedding = model.encode(query, convert_to_numpy=True)
    scores = cosine_similarity([query_embedding], paper_embeddings)[0]
    indices = np.argsort(-scores)[:top_k]
    return [cord_uids[i] for i in indices]

## Evaluation

In [19]:
model = SentenceTransformer("MiniLM-mnr-ep3")
df_query_train['neural_topk'] = df_query_train['tweet_text'].apply(lambda x: get_top_cord_uids_neural(x, model))
df_query_dev['neural_topk'] = df_query_dev['tweet_text'].apply(lambda x: get_top_cord_uids_neural(x, model))

results_train = get_performance_mrr(df_query_train, 'cord_uid', 'neural_topk')
results_dev = get_performance_mrr(df_query_dev, 'cord_uid', 'neural_topk')
# Printed MRR@k results in the following format: {k: MRR@k}
print(f"Results on the train set: {results_train}")
print(f"Results on the dev set: {results_dev}")

Results on the train set: {1: 0.44962265618921654, 5: 0.5045151585881377, 10: 0.5045151585881377}
Results on the dev set: {1: 0.2914285714285714, 5: 0.32603571428571426, 10: 0.32603571428571426}


In [ ]:
model_mpnet = SentenceTransformer("Mpnet-mnr-ep3")
df_query_train['neural_topk'] = df_query_train['tweet_text'].apply(lambda x: get_top_cord_uids_neural(x, model_mpnet))
df_query_dev['neural_topk'] = df_query_dev['tweet_text'].apply(lambda x: get_top_cord_uids_neural(x, model_mpnet))

results_train = get_performance_mrr(df_query_train, 'cord_uid', 'neural_topk')
results_dev = get_performance_mrr(df_query_dev, 'cord_uid', 'neural_topk')
# Printed MRR@k results in the following format: {k: MRR@k}
print(f"Results on the train set: {results_train}")
print(f"Results on the dev set: {results_dev}")

## Cosine with hard negatives

In [20]:
train_examples = []

for index, row in df_query_train.iterrows():
    paper_id = row['cord_uid']
    tweet_text = row['tweet_text']
    pos_title = df_collection.loc[df_collection['cord_uid'] == paper_id, 'title'].values[0]
    pos_abstract = df_collection.loc[df_collection['cord_uid'] == paper_id, 'abstract'].values[0]
    pos_text = f"{pos_title}. {pos_abstract}"
    input_example = InputExample(texts=[tweet_text, pos_text], label=1.0)
    train_examples.append(input_example)

    hard_negatives = [cid for cid in row['bm25_topk'] if cid!=row['cord_uid']]
    for neg_id in hard_negatives:
        neg_title = df_collection.loc[df_collection['cord_uid'] == neg_id, 'title'].values[0]
        neg_abstract = df_collection.loc[df_collection['cord_uid'] == neg_id, 'abstract'].values[0]
        neg_text = f"{neg_title}. {neg_abstract}"
    
        input_example = InputExample(texts=[tweet_text, neg_text], label=0.0)
        train_examples.append(input_example)

pd.DataFrame(train_examples).to_pickle("train_examples_5_hard_negatvies.pkl")

In [21]:
from torch.utils.data import DataLoader
from sentence_transformers import losses

model = SentenceTransformer('all-MiniLM-L6-v2') # smaller faster transformer

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.CosineSimilarityLoss(model=model)

model_filename = "MiniLM-cos-ep3"

if True:
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=3,
        warmup_steps=100,
        show_progress_bar=True
    )
    model.save(model_filename)

model = SentenceTransformer(model_filename)

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
ERROR: ld.so: object '/opt/conda/lib/libmkl_def.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_avx2.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_core.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_intel_lp64.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/opt/conda/lib/libmkl_intel_thread.so' from LD_PRELOAD

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.123600
1000,0.112500
1500,0.109600
2000,0.104800
2500,0.107800
3000,0.104700
3500,0.103300
4000,0.104100
4500,0.103300
5000,0.103500


## Evaluation

In [22]:
df_query_train['neural_topk'] = df_query_train['tweet_text'].apply(lambda x: get_top_cord_uids_neural(x, model))
df_query_dev['neural_topk'] = df_query_dev['tweet_text'].apply(lambda x: get_top_cord_uids_neural(x, model))

results_train = get_performance_mrr(df_query_train, 'cord_uid', 'neural_topk')
results_dev = get_performance_mrr(df_query_dev, 'cord_uid', 'neural_topk')
# Printed MRR@k results in the following format: {k: MRR@k}
print(f"Results on the train set: {results_train}")
print(f"Results on the dev set: {results_dev}")

Results on the train set: {1: 0.44713296506652145, 5: 0.5053048574911175, 10: 0.5053048574911175}
Results on the dev set: {1: 0.305, 5: 0.3470595238095238, 10: 0.3470595238095238}


# Test set

In [40]:
model = SentenceTransformer("MiniLM-mnr-ep3")
PATH_QUERY_TEST_DATA = "subtask4b_query_tweets_test.tsv"
df_query_test = pd.read_csv(PATH_QUERY_TEST_DATA, sep = '\t')

df_query_test['neural_topk'] = df_query_test['tweet_text'].apply(lambda x: get_top_cord_uids_neural(x, model))
df_query_test['preds'] = df_query_test['neural_topk'].apply(lambda x: x[:5])
df_query_test[['post_id', 'preds']].to_csv('predictions.tsv', index=None, sep='\t')